# Gaussian mean and scale controls

This development study separates calibration choices from mean-versus-scale selection in [#629](https://github.com/fcela/mboostrs/issues/629) and [#630](https://github.com/fcela/mboostrs/issues/630). A standardized-score experiment holds every fitted mean and scale fixed. Three later conditional-scale prototypes test whether a scale candidate can win by changing its mean fit. None met its frozen promotion gate, so the Gaussian default and scale-adoption rule remain unchanged.

Each process has ten independent development draws, 1,000 analysis observations, a common 500/500 fitting/calibration split and 2,000 independent evaluation observations. All contrasts reuse the same per-process inputs. Processes share random streams and are never pooled for uncertainty. The 1,900 method/draw records include 100 oracle-only diagnostics; they are not 1,900 independent datasets. Confirmation for these rejected candidates remains ungenerated.

Five independent Uniform(−1,1) predictors are observed. Smooth means are $\sin(2\pi x_1)+0.5x_2$; sharp means are $3\operatorname{sign}(x_1-a)\operatorname{sign}(x_2-b)+x_3$ at $(a,b)=(0,0)$ or $(0.25,-0.2)$; the null mean is zero. Constant sharp standard deviation is 0.5, other constant controls use 1, and varying standard deviations are $0.3+1.5|x_1|$, $0.3+1.5\exp(-8x_1^2)$ or $\exp(0.7x_1)$. Errors are Gaussian except the constant-variance heavy-tail control, Student-t3 divided by $\sqrt{3}$. Extra columns are noise where the generating mean/scale omit them.

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import polars as pl
from great_tables import GT, md

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "Cargo.toml").exists())
sys.path.insert(0, str(root / "python"))
sys.path.insert(0, str(root))
from benches.evidence_float_codec import decode_record

base = root / "docs/book/benchmarks"
directory = base / "gaussian_scale_calibration"
rows = decode_record(json.loads((directory / "development_summary.json").read_text()))["rows"]
gate = json.loads((directory / "development_gate.json").read_text())
conditional = json.loads((directory / "CONDITIONAL_DEVELOPMENT_SUMMARY.json").read_text())

In [ ]:
def headline(records, title):
    best = {}
    for panel in {r["panel"] for r in records}:
        eligible = [r for r in records if r["panel"] == panel and r.get("coverage_eligible") and not r["failures"]]
        for key in ("width", "true_mean_mse", "interval_score"):
            best[panel, key] = min((r[key] for r in eligible), default=None)

    def value(record, key):
        if key not in record:
            return "diagnostic only"
        text = f"{record[key]:.3f}"
        if key == "width" and record.get(key + "_se") is not None:
            text += f" ± {record[key + '_se']:.3f}"
        if record.get("coverage_eligible") and record[key] == best.get((record["panel"], key)):
            text = "**" + text + "**"
        return text

    frame = pl.DataFrame([{"Process": r["panel"], "Method": r["method"].replace("_", " "),
        "Width": value(r,"width"), "Coverage": value(r,"coverage"), "Mean MSE": value(r,"true_mean_mse"),
        "Interval score": value(r,"interval_score"), "Failures": r["failures"]} for r in records])
    # Coverage eligibility prevents a narrow under-covering procedure from being marked best.
    return (GT(frame,rowname_col="Method",groupname_col="Process").tab_stubhead("Method")
        .tab_header(title=title,subtitle="Ten independent draws per process; smaller width, latent-mean MSE and score are better")
        .fmt_markdown(columns=["Width","Mean MSE","Interval score"])
        .tab_source_note(md("Width is real response-unit length, mean ± Monte Carlo SE across draws. Mean MSE compares "
            "the reported point with the generating mean, in squared response units. CQR points are pre-calibration "
            "band midpoints. Score is the 90% endpoint interval-score expression. Bold marks complete eligible "
            "minima within process, not a chosen new method. Oracle residual diagnostics produce no prediction band. "
            "Source: development_summary.json, which summarize_gaussian_scale_calibration.py computes from every "
            "complete per-draw archive; no successful-only aggregates.")))

In [ ]:
headline([r for r in rows if r["panel"] == "smooth_bell"], "Every fixed method on bell-scale development")

## Calibration scores

The original distributional route starts with $\mu\pm z_{0.95}\sigma$ and adds a common calibration correction to the endpoints. The standardized score is $|Y-\mu|/\sigma$, with endpoints $\mu\pm q\sigma$ after the usual conformal order statistic. Its cutoff is dimensionless; the legacy additive cutoff is in response units. Positive finite scales are required. Both constructions calibrate a fixed proper-training fit, with marginal coverage under exchangeability; neither identifies a true noise mechanism.

Four fixed fitting tracks supply paired scores: ordinary discovery; supplied varying GaussianLSS at 100 updates per parameter; the same supplied formulas with diagonal stopping tuned on five fitting folds, start 100/cap 400; and a supplied constant-scale diagnostic. Smooth supplied mean is `bbs(x1,df=4)+bols(x2)`, sharp supplied mean is the pair depth-3 SSE tree plus `bols(x3)`, and null mean is `1`. Varying supplied scale is `bbs(x1,df=4)`, constant scale `1`. Rate is 0.1. Every score pair reuses exactly the same fitted mean, scale and point vector.

The other fixed methods are spread-disabled discovery, supplied CQR at 1,600 updates/rate 0.1, the labelled oracle-mean residual diagnostic, and the same five CatBoost/LightGBM/sklearn controls used in the count studies. CatBoost uses 500 trees/depth 6/rate 0.05; LightGBM 300 trees/15 leaves/rate 0.05; sklearn CQR 300 trees/depth 3/rate 0.05. These five-predictor controls are distinct from the original two-predictor #630 task.

In [ ]:
required = [r for r in rows if r["panel"] in ("smooth_bell","smooth_monotone")
            and r["method"] in ("discovery_normalized","supplied_fixed_normalized")]
frame = pl.DataFrame([{"Process":r["panel"],"Method":r["method"],"Coverage":r["coverage"],
    "Width excess %":r["width_excess"],"Upper 95%":r["width_upper95"],"Upper target":r["upper_target"]}
    for r in required])
(GT(frame,rowname_col="Method",groupname_col="Process").tab_stubhead("Required information track")
 .tab_header(title="Standardized-score development gate",subtitle="Paired percent width excess over prespecified LightGBM CQR; upper target ≤ 1%")
 .fmt_number(columns=["Coverage","Width excess %","Upper 95%"],decimals=3)
 .tab_source_note("Coverage eligibility is [0.88,0.92] at nominal0.90. The upper bound uses Student-t across ten paired "
                  "independent draws; these remain development results. All five reference comparisons and adjusted bounds are retained."))

In [ ]:
assert not gate["targets_pass"] and not gate["promote_default"] and not gate["confirmation_generated"]
print("The standardized-score candidate is not promoted; its reserved confirmation draws remain unused.")

The standardized correction increases width on the bell control at comparable marginal coverage. The gate fails in the supplied fixed track and the bell data-only upper bound. The experimental `score="gaussian_normalized"` implementation is retained for numerical comparison; the Gaussian automatic route keeps its existing central-band calibration. R exposes the explicit experimental score for GaussianLSS. Its numerical implementation is shared Rust, with no new Python/R scoring formula.

## Public score comparison

This example repeats one frozen supplied bell input to demonstrate the fixed-fit contrast. It does not create additional confirmation evidence. Fitting, nuisance estimation and stopping choices precede calibration.

In [ ]:
import mboostrs as mb
assert Path(mb.__file__).resolve().is_relative_to(root)
from benches.study_sharp_mean_controls import draw
from benches.study_discovery_calibration import partition

analysis, _, _ = draw("smooth_bell", 6293100, 1000)
evaluation, _, _ = draw("smooth_bell", 6393100, 2000)
fitting, calibration, partition_record = partition(analysis, 6293100)
model = mb.gamboostLSS({"mu":"y ~ bbs(x1, df=4) + bols(x2)","sigma":"y ~ bbs(x1, df=4)"},
    fitting,family=mb.GaussianLSS(),control=mb.boost_control(100))
legacy = mb.conformal_interval(model,evaluation.drop("y"),calibration=calibration,
                               score="absolute",extrapolation="clip")
standardized = mb.conformal_interval(model,evaluation.drop("y"),calibration=calibration,
                                     score="gaussian_normalized",extrapolation="clip")
np.testing.assert_array_equal(legacy.fitted_mean,standardized.fitted_mean)
comparison = pl.DataFrame({"Score":["Central band + additive","Standardized"],
    "Width":[float(legacy.width.mean()),float(standardized.width.mean())],
    "Coverage":[float(legacy.contains(evaluation["y"]).mean()),float(standardized.contains(evaluation["y"]).mean())]})
(GT(comparison).tab_header(title="Same fitted model, two calibration scores",subtitle="One repeated development input; response units")
 .fmt_number(columns=["Width","Coverage"],decimals=3)
 .tab_source_note("Mean predictions are identical. This display illustrates the transformation; it does not select a score or promote a default."))

## Conditional scale comparisons

A joint Gaussian location-scale comparison can change the mean fit while assessing a varying scale. Under an incompletely fitted sharp mean, a scale candidate can therefore improve prediction without representing a true noise-variance driver. The baseline development controls select varying scale on three of ten axis-constant and two of ten shifted-constant draws, while detecting none of their genuinely varying sharp counterparts.

Version 1 fits one Gaussian mean at 200 updates on each fitting fold and shares its predictions across constant and nominated scale candidates. It fits absolute mean residuals with location fixed at zero and 200 scale updates, using Gaussian likelihood for the scale comparison. Symmetry makes signed and absolute residuals equivalent at zero location. The existing best-candidate/two-SE gate and final joint fitting remain unchanged.

Version 2 tunes each outer-fold Gaussian mean on five inner fitting folds, start 100/cap 400, and nominates scale from assembled out-of-fold residuals. It shares those mean fits across scale candidates. Ten null controls record a prototype-only refusal to tune an intercept-only path; their delivered fallback predictions remain included, and the scale diagnostic is unavailable there.

Version 3 guards intercept-only means and uses a fixed Student-t reference with four degrees of freedom for conditional scale fitting and scoring. Location remains zero and degrees of freedom remain four; only scale is fitted. This bounds the scale gradient's leverage from very large mean residuals. Returned prediction models still use the existing Gaussian pipeline after a supported adoption. These implementations remain development-only code in `benches`; production numeric implementation, both-language bindings and complete cost admission would precede any confirmation.

In [ ]:
versions = {"conditional_scale_comparison":"Fixed mean / Gaussian scale",
            "conditional_scale_comparison_v2":"Cross-fitted tuned mean",
            "conditional_scale_comparison_v3":"Cross-fitted mean / t4 scale"}
records = []
for r in rows:
    if r["method"] == "discovery_central":
        records.append({"Process":r["panel"],"Method":"Current discovery","Scale selected":f"{r['scale_adoptions']}/10",
            "Stage refusals":0,"Mean MSE":r["true_mean_mse"],"Width":r["width"],"Coverage":r["coverage"]})
for r in conditional:
    evaluated = not r["prediction_failures"] and not r["spread_refusals"]
    records.append({"Process":r["panel"],"Method":versions[r["version"]],
        "Scale selected":f"{r['adoptions']}/10" if evaluated else "diagnostic unavailable",
        "Stage refusals":r["spread_refusals"],"Mean MSE":r.get("true_mean_mse"),"Width":r.get("width"),"Coverage":r.get("coverage")})
(GT(pl.DataFrame(records),rowname_col="Method",groupname_col="Process").tab_stubhead("Comparison")
 .tab_header(title="Scale decisions and delivered predictions",subtitle="Ten paired development draws per process; smaller MSE/width are better at eligible coverage")
 .fmt_number(columns=["Mean MSE","Width","Coverage"],decimals=3)
 .tab_source_note("Scale selected means a varying-scale term was adopted. On constant-variance processes it is a false "
                  "structural attribution, not necessarily a prediction regression. Prediction metrics include every returned "
                  "fallback; they are not successful-only averages. Version2 null scale decisions are unavailable because "
                  "all ten diagnostic stages refused. Counts are descriptive development outcomes, not population error bounds."))

The robust conditional comparison selects no varying scale on the tested constant-variance controls and detects all smooth varying-scale controls. Its axis-varying sharp control reaches only five of ten adoptions, below the frozen requirement of 90% observed recall. The candidate is therefore not promoted. The conditional experiment removes one joint-mean comparison ambiguity but does not complete noise attribution. The sharp-control confirmation cohort was later used by the mean-adequacy guard confirmation, not by any conditional candidate; the section below states what that confirmation showed for a sharp mean with a smooth scale.

## Sharp mean with a smooth scale: a stated limitation

On two processes from one generator, the default did not detect a smooth scale under a sharp mean. Both processes use the scale 0.3 + 1.5|x1|, the same scale as the smooth_absolute control. Their means are 3·sign(x1)·sign(x2) + x3 (axis_varying) and 3·sign(x1−0.25)·sign(x2+0.2) + x3 (shifted_varying). In the sharp-mean confirmation battery on reserved seeds 6294100–6294139 (40 draws per process), discovery adopted a varying scale on 0 of 40 draws of axis_varying and 0 of 40 draws of shifted_varying, while adopting on 40 of 40 draws of each smooth varying-scale process (absolute, bell and monotone) and on 0 of 40 draws of every constant-variance process. The same sharp processes read 0 of 10 in the development inventory, before the mean-adequacy guard, so the guard did not cause the miss. A diagnostic recorded with the decision shows the failure at the nomination and screening steps, on residuals of the 200-update screening mean fit; the scale term that fails there is the same smooth term the smooth processes adopt.

This is an accepted limitation of the default, decided in the [sharp-scale decision record](sharp_mean_controls/SHARP_SCALE_DECISION.md). Marginal coverage on the two sharp varying-scale processes remains 0.901, inside the [0.88, 0.92] eligibility band; what is lost is scale attribution, and coverage split by true-scale tercile is unbalanced (0.963 low / 0.805 high on axis_varying; 0.978 / 0.793 on shifted_varying, against a nominal 0.90). A supplied GaussianLSS sigma formula, or a mandatory spread column in discovery, still fits a varying scale on such data. A future claim that the default recovers a smooth scale under a sharp mean requires a candidate frozen on development draws and confirmed on a fresh cohort, under bars stated in advance in the decision record.

## Complete score controls

In [ ]:
headline(rows, "All fixed score-development methods and processes")

## Evidence and reproduction

The score protocol is frozen at `9ddd025` and its implementation/runner at `c9a9dda`. Conditional versions use their separately frozen protocols and runners: version 1 `1c4ee18`/`e840894`, version 2 `495361d`/`a8a8fad`, version 3 `46646b8`/`07cd0e8`. The source validator verifies all 1,900 archive and runner identities and identical per-process input hashes. Model-fitting Rust modules are identical across these contrasts; the added standardized-score functions are recorded separately in the native build identity.

All score-study calibration calls and competitor predictions/scores are retained. Conditional version 1 keeps full conditional fold means, scale predictions and scores plus returned intervals; full calibration call capture starts in version 2. Earlier missing call vectors are not filled in retrospectively. The initial conditional smoke assertion also exposed optional tree-refit refusals, tracked separately in [#671](https://github.com/fcela/mboostrs/issues/671); those logs remain visible and did not identify a scale-stage failure.

The [budget clarification](default_stopping_ceilings.md) distinguishes the public default mean ceiling of 400 from the LSS ceiling of 1,600 per parameter. Earlier protocol shorthand is preserved with that explicit interpretation. Supplied tuned controls retain their stated explicit cap 400. The external-version addendum is explicitly post-run inventory, rather than a rewritten measurement-time record.

Independent score checks cover inversion, positive scale boundaries, affine changes, internal/external calibration equivalence and legacy routing. All 291 integrated Python checks, 776 Rust core tests, 14 integration tests and seven Rust doc tests pass, as does strict Clippy. Fixed-input Python/R score/band differences are zero; public fitted differences are $8.88\times10^{-16}$. Numerical validation establishes implementation agreement, not competitive efficiency.

From the matching source checkout and repository environment:

```bash
PYTHONPATH=python:. .venv/bin/python benches/study_gaussian_scale_calibration.py \
  --phase development --panels smooth_bell --output /tmp/gaussian-score-bell.jsonl.xz
PYTHONPATH=python:. .venv/bin/python benches/summarize_gaussian_scale_calibration.py --phase development
PYTHONPATH=python:. .venv/bin/python benches/summarize_conditional_scale_development.py
```

The conditional runners accept `--panel` and a new `--output` filename. They refuse to overwrite evidence. Future production or confirmation work uses a new declared source freeze and does not relabel these development results. Costs are descriptive under concurrent jobs; memory was not measured.

In [ ]:
folders = ["gaussian_scale_calibration", "conditional_scale_comparison", "conditional_scale_comparison_v2", "conditional_scale_comparison_v3"]
# Per-draw archives are release assets listed in raw_archives.json; the bundle holds the files this page reads or cites.
artifacts = [folder+"/"+p.name for folder in folders for p in sorted((base/folder).iterdir())
             if p.is_file() and not p.name.endswith((".jsonl.xz", ".jsonl.gz"))]
(GT(pl.DataFrame({"Artifact":artifacts})).tab_header(title="Bundled development evidence",
    subtitle="Protocols, run metadata, summaries and validation")
 .tab_source_note("The four experiments reuse development inputs. The data download is not independent confirmation evidence. "
                  "Per-draw archives (*.jsonl.xz) are assets of the benchmark-raw-archives-1 GitHub release, listed in raw_archives.json, and are not bundled."))